In [15]:
import numpy as np
import sympy as sp
from scipy.integrate import quad
import matplotlib.pyplot as plt
import math
import time
from scipy.linalg import hankel
from tqdm import tqdm
import pandas as pd
import os
import warnings
import glob

In [16]:
#parametros
M_min = 0.1
M_max = 10.0
num_M = 10000

M_array = np.linspace(M_min, M_max, num_M)

tmax = 2000
rsv = 4
numMax = 20001
L = 2
aGaussiana, bGaussiana = rsv, 3
perturbation = (aGaussiana, bGaussiana)
rc = 100.0

In [ ]:
def genelist(a, b, n):
    return np.linspace(a, b, int(n))

def V_potencial(rlist, L, M):
        return np.where(
            rlist <= rc,
            (1.0 - 2 * M / rlist) * (L * (L + 1) / rlist**2 - 6 * M / rlist**3),
            0.0,
        )
        # return (1.0 - 2 * M / rlist) * (L * (L + 1) / rlist**2 - 6 * M / rlist**3) sem descontinuidade



In [18]:
def findrvalue_vetorizado(rstarlist, M, tol=1e-14, max_iter=100):
    """
    Resolve r* = r + 2M*ln(r/(2M) - 1) para r, dado um array de r*,
    usando Newton-Raphson vetorizado (substitui a bisseção ponto a ponto).
    """
    # chute inicial: para r grande, r* ≈ r (assintoticamente)
    r = np.maximum(rstarlist, 2 * M * 1.001)
    w = np.log((r/(2 * M)) - 1)

    for it in range(max_iter):
        f_w = 2 * M * (1 + w + np.exp(w)) - rstarlist
        f_w_linha = 2 * M * (1 + np.exp(w))

        w_new = w - f_w / f_w_linha

        delta = np.max(np.abs(w_new - w))
        # analisa a diferença maxima entre os valores antigos e novos para todos os elementos do array
        w = w_new

        if delta < tol:
            break

    r = 2 * M * (1 + np.exp(w))
    return r

In [19]:
def getEvo(tmax, rsv, numMax, M, L, pertubation_params):
    av, bv = pertubation_params

    rsmax = 2 * tmax
    rstarlist = genelist(rsv - rsmax/2.0, rsv + rsmax/2.0, numMax)
    
    hr = float(np.diff(rstarlist[:2])[0])
    ht =0.5 * hr 
    # print(f"ht: {ht}")

    inicio = time.time()
    rlist0 = findrvalue_vetorizado(rstarlist, M)
    tempo_execucao  = time.time() - inicio
    # print(f"Tempo de execução para encontrar rlist0: {tempo_execucao:.6f} segundos")

    rlist = rlist0

    #Verificação e Aviso
    # if np.any(rlist <= rc):
    #     # Localiza o índice na malha mais próximo do valor exato de rc
    #     idx_rc = np.argmin(np.abs(rlist - rc))
    #     r_proximo = rlist[idx_rc]

    #     print(f"AVISO: O ponto rc = {rc} foi atingido pela malha!")
    #     print(
    #         f"   -> Ponto no grid mais próximo: r = {r_proximo:.14f} (Índice: {idx_rc})"
    #     )
    #     print(f"   -> Ponto r* correspondente: r* = {rstarlist[idx_rc]:.6f}")
    # else:
    #     print(
    #         f"O valor rc = {rc} não foi atingido na malha (r_min = {rlist.min():.4f})"
    #     )


    Vv = V_potencial(rlist, L, M)

    ii = 1
    psi0 = np.zeros(len(rlist))
    psi1 = ht * np.exp(-((rstarlist - av)**2) / (2 * (bv**2)))
    length = int((numMax - 1) / 2)
    psi_collect = np.zeros(length)

    inicio_loop = time.time()
    for ii in range(1, length+1):
        Vv = Vv[1:-1]
        psi2 = (
            2 * psi1[1:-1] 
            - psi0[1:-1] 
            - (ht**2) * Vv * psi1[1:-1] 
            + (ht**2 / hr**2) * (psi1[:-2] - 2 * psi1[1:-1] + psi1[2:])
        )
        
        psi0 = psi1[1:-1]
        psi1 = psi2

        psi_collect[ii-1] = psi1[length - ii] # armazenando o valor de psi no ponto central da malha para cada passo de tempo
        

    tempo_execucao_loop  = time.time() - inicio_loop
    # print(f"Tempo de execução do loop: {tempo_execucao_loop:.6f} segundos")
    return ht, psi_collect

In [20]:
output_dir = "dataset_QNM"
os.makedirs(output_dir, exist_ok=True)


Ms_com_warning = []

for i, M in enumerate(tqdm(M_array, desc="Processando massas")):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")  # garante que todo warning seja capturado, mesmo repetido
        
        ht, psi = getEvo(tmax, rsv, numMax, M, L, perturbation)
        
        if len(w) > 0:
            Ms_com_warning.append((M, i, [str(warn.message) for warn in w]))
            print(f"⚠️ M={M:.4f} (i={i}) disparou {len(w)} warning(s)")

    filename = os.path.join(output_dir, f"sample_{i:06d}.npz")
    np.savez(filename, ht=ht, psi=psi, M=M)

print(f"\nTotal de Ms com warning: {len(Ms_com_warning)}")
for M, i, msgs in Ms_com_warning:
    print(f"M={M:.4f} (i={i}): {msgs[0]}")

Processando massas: 100%|██████████| 10000/10000 [1:55:02<00:00,  1.45it/s]


Total de Ms com warning: 0


In [21]:
files = sorted(glob.glob(os.path.join(output_dir, "*.npz")))
files = np.array(files)  # para poder indexar com arrays booleanos/índices

n_total = len(files)
print(f"Total de amostras: {n_total}")

# 2) Embaralha os ÍNDICES (não os dados em si, só a ordem)
rng = np.random.default_rng(seed=42)  # seed fixa = reprodutível
indices = np.arange(n_total)
rng.shuffle(indices)

# 3) Define os tamanhos de cada split
n_train = int(0.70 * n_total)
n_test  = int(0.20 * n_total)
n_val   = n_total - n_train - n_test  # o resto vai para validação (garante que soma bate exatamente)

# 4) Separa os índices embaralhados em blocos, SEM sobreposição
idx_train = indices[:n_train]
idx_test  = indices[n_train : n_train + n_test]
idx_val   = indices[n_train + n_test :]

print(f"Treino: {len(idx_train)} | Teste: {len(idx_test)} | Validação: {len(idx_val)}")

# 5) Pega os arquivos correspondentes a cada split
files_train = files[idx_train]
files_test  = files[idx_test]
files_val   = files[idx_val]

Total de amostras: 10000
Treino: 7000 | Teste: 2000 | Validação: 1000


In [22]:
def carregar_split(lista_arquivos):
    psi_list, M_list = [], []
    for f in lista_arquivos:
        data = np.load(f)
        psi_list.append(data["psi"])  # ajuste a chave conforme você salvou
        M_list.append(data["M"])
    return np.array(psi_list), np.array(M_list)

psi_train, M_train = carregar_split(files_train)
psi_test,  M_test  = carregar_split(files_test)
psi_val,   M_val   = carregar_split(files_val)

save_dir = "dataset_splits"
os.makedirs(save_dir, exist_ok=True)
# Salva cada split consolidado separadamente
np.savez(os.path.join(save_dir, "dataset_train.npz"), psi=psi_train, M=M_train)
np.savez(os.path.join(save_dir, "dataset_test.npz"),  psi=psi_test,  M=M_test)
np.savez(os.path.join(save_dir, "dataset_val.npz"),   psi=psi_val,   M=M_val)

![alt text](imagens\fluxograma.png "Title")

![alt text](imagens\demonstracaoH.jpg "Title")